<a href="https://colab.research.google.com/github/tribobbyjones/obd-project/blob/main/OBD_Final_Exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 OBD-II Agentic RAG System — Final Exam

**Five distinct agent architectures on a shared OBD-II automotive knowledge base.**

| Cell | Agent | Architecture |
|------|-------|-------------|
| 2 | Setup | Install + imports |
| 3 | Knowledge Base | ChromaDB + synthetic OBD docs |
| 4 | Shared Tools | 4 domain search functions |
| 5 | Agent 1 | ReAct (Thought→Action→Observation) |
| 6 | Agent 2 | Debate (Advocate→Critic→Judge) |
| 7 | Agent 3 | Supervisor (Draft→Evaluate→Retry) |
| 8 | Agent 4 | Multi-Agent Parallel (asyncio) |
| 9 | Agent 5 | Meta-Intelligent Router |
| 10 | Launch | All agents in one tabbed Gradio UI |

> **Note:** No physical OBD adapter needed. All agents work from the synthetic knowledge base seeded in Cell 3.

In [ ]:
# ── CELL 2: INSTALL & CONFIGURE ─────────────────────────────
!pip install -q openai chromadb sentence-transformers gradio nest_asyncio

import os, re, json, uuid, time, asyncio, threading, warnings
from datetime import datetime
warnings.filterwarnings("ignore", category=DeprecationWarning)

import gradio as gr
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from openai import OpenAI

# ── Set your OpenAI API key ──────────────────────────────────
from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")


os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

MODEL           = "gpt-4o-mini"
TEMPERATURE     = 0.2
REQUEST_TIMEOUT = 30.0
MAX_TOKENS      = 1024

openai_client = OpenAI(api_key=OPENAI_API_KEY, timeout=REQUEST_TIMEOUT)
print("✅ Setup complete.")

✅ Setup complete.


In [ ]:
# ── CELL 3: PID REGISTRY & KNOWLEDGE BASE ───────────────────
# Synthetic OBD-II knowledge documents seeded into ChromaDB.
# This is the shared knowledge base used by all 5 agents.

PID_REGISTRY = {
    "RPM":                  {"label": "Engine RPM",              "unit": "RPM",   "normal": (600, 5000),    "warn": 5500,  "critical": 6500,  "warn_direction": "high",  "note": "High idle >1000 RPM when warm can indicate IAC fault or vacuum leak."},
    "SPEED":                {"label": "Vehicle Speed",           "unit": "km/h",  "normal": None,           "warn": None,  "critical": None,  "warn_direction": "high",  "note": ""},
    "ENGINE_LOAD":          {"label": "Engine Load",             "unit": "%",     "normal": (0, 85),        "warn": 90,    "critical": 100,   "warn_direction": "high",  "note": "Sustained load >85% under normal driving may indicate mechanical restriction."},
    "COOLANT_TEMP":         {"label": "Engine Coolant Temp",     "unit": "\u00b0C",    "normal": (70, 100),      "warn": 105,   "critical": 110,   "warn_direction": "high",  "note": "Normal 80-95\u00b0C. Above 105\u00b0C risks head gasket damage. Stop driving if >110\u00b0C."},
    "INTAKE_TEMP":          {"label": "Intake Air Temp",         "unit": "\u00b0C",    "normal": None,           "warn": 60,    "critical": 75,    "warn_direction": "high",  "note": "Excessively hot intake air reduces power and risks detonation."},
    "OIL_TEMP":             {"label": "Engine Oil Temp",         "unit": "\u00b0C",    "normal": (80, 120),      "warn": 130,   "critical": 150,   "warn_direction": "high",  "note": "Oil >130\u00b0C degrades rapidly. Check oil level and cooling system."},
    "SHORT_FUEL_TRIM_1":    {"label": "Short Fuel Trim B1",      "unit": "%",     "normal": (-10, 10),      "warn": 15,    "critical": 25,    "warn_direction": "high",  "note": "Positive = ECU adding fuel (lean). Negative = removing fuel (rich)."},
    "LONG_FUEL_TRIM_1":     {"label": "Long Fuel Trim B1",       "unit": "%",     "normal": (-10, 10),      "warn": 15,    "critical": 25,    "warn_direction": "high",  "note": "Persistent lean corrections >+15% indicate chronic vacuum leak or injector issue."},
    "FUEL_LEVEL":           {"label": "Fuel Level",              "unit": "%",     "normal": None,           "warn": 10,    "critical": 5,     "warn_direction": "low",   "note": "Low fuel warning."},
    "MAF":                  {"label": "Mass Air Flow",           "unit": "g/s",   "normal": None,           "warn": None,  "critical": None,  "warn_direction": "high",  "note": "Low MAF at high load can indicate dirty/failed sensor or air restriction."},
    "THROTTLE_POS":         {"label": "Throttle Position",       "unit": "%",     "normal": None,           "warn": None,  "critical": None,  "warn_direction": "high",  "note": ""},
    "TIMING_ADVANCE":       {"label": "Timing Advance",          "unit": "\u00b0",     "normal": None,           "warn": None,  "critical": None,  "warn_direction": "high",  "note": "Retarded timing under load may indicate knock sensor activity."},
    "O2_B1S1":              {"label": "O2 Sensor B1S1",          "unit": "V",     "normal": None,           "warn": None,  "critical": None,  "warn_direction": "high",  "note": "Upstream sensor — should oscillate 0.1-0.9V in closed loop."},
    "O2_B1S2":              {"label": "O2 Sensor B1S2",          "unit": "V",     "normal": None,           "warn": None,  "critical": None,  "warn_direction": "high",  "note": "Downstream sensor — should be steady if catalyst is healthy."},
    "CONTROL_MODULE_VOLTAGE":{"label": "Module Voltage",         "unit": "V",     "normal": (13.5, 14.8),   "warn": 12.0,  "critical": 11.0,  "warn_direction": "low",   "note": "Below 13.5V at idle indicates charging system fault. Below 11V is critical."},
    "EGR_ERROR":            {"label": "EGR Error",               "unit": "%",     "normal": (-10, 10),      "warn": 20,    "critical": 30,    "warn_direction": "high",  "note": "Large EGR error indicates valve stuck open or closed."},
    "CATALYST_TEMP_B1S1":   {"label": "Catalyst Temp B1S1",      "unit": "\u00b0C",    "normal": (400, 800),     "warn": 850,   "critical": 950,   "warn_direction": "high",  "note": "Catalyst overtemp can indicate misfires dumping raw fuel into exhaust."},
    "DISTANCE_W_MIL":       {"label": "Distance With MIL On",    "unit": "km",    "normal": None,           "warn": None,  "critical": None,  "warn_direction": "high",  "note": "Non-zero value means check engine light has been on during driving."},
}

embedding_fn  = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.Client()   # in-memory for Colab

def _get_or_create(name):
    try:
        return chroma_client.get_collection(name=name, embedding_function=embedding_fn)
    except Exception:
        return chroma_client.create_collection(name=name, embedding_function=embedding_fn)

kb_engine     = _get_or_create("obd_engine")
kb_fuel       = _get_or_create("obd_fuel")
kb_electrical = _get_or_create("obd_electrical")
kb_dtc        = _get_or_create("obd_dtc")

def _seed(collection, docs, ids):
    if collection.count() == 0:
        collection.add(documents=docs, ids=ids, metadatas=[{"source": "seed"}]*len(docs))

_seed(kb_engine, [
    "Engine RPM normal range is 600-5000 RPM. Idle RPM above 1000 when warm suggests an IAC fault or vacuum leak.",
    "Engine coolant temperature should stay between 70-100°C. Readings above 105°C risk head gasket damage. Stop driving if >110°C.",
    "Engine oil temperature normal range is 80-120°C. Oil above 130°C degrades rapidly; check oil level and cooling system.",
    "Engine load above 85% sustained under normal driving may indicate a clogged air filter or failing turbocharger.",
    "Intake air temperature above 60°C reduces engine power. Above 75°C is critical.",
    "Timing advance retardation under load indicates the knock sensor detecting detonation and pulling timing.",
    "Catalyst temperature B1S1 should be 400-800°C. Above 850°C warning; above 950°C critical — misfires suspected.",
    "MAF sensor readings low under high load suggest a dirty sensor or air intake restriction.",
    "OBD-II Mode 06 readiness monitors show whether emissions components have completed self-tests since last DTC clear.",
    "IAC valve failures cause erratic or high idle RPM and may set P0505 codes.",
], [f"eng_{i}" for i in range(10)])

_seed(kb_fuel, [
    "Short fuel trim (STFT) oscillates in real time. Values beyond ±10% indicate lean or rich correction.",
    "Long fuel trim (LTFT) above +15% persistently suggests a vacuum leak, clogged injector, or failing MAF sensor.",
    "Long fuel trim below -15% persistently suggests a rich condition: leaking injectors, high fuel pressure, or faulty coolant temp sensor.",
    "Lambda of 1.0 is stoichiometric. Above 1.05 = lean; below 0.95 = rich.",
    "O2 sensor B1S1 should oscillate 0.1-0.9V in closed loop. Flat signal at 0.45V = failed sensor.",
    "O2 sensor B1S2 (downstream) holds ~0.6-0.7V if catalytic converter is functioning.",
    "Fuel level below 10% triggers a low fuel warning. Below 5% risks fuel pump overheating.",
    "EGR error above 20% indicates the EGR valve is not responding to commands, affecting NOx emissions.",
    "EVAP purge valve duty cycle should vary during normal operation. A stuck valve causes P0441 or P0446.",
    "Fuel injector balance testing compares fuel trim across cylinders. Misfire plus rich trim on one bank = stuck-open injector.",
], [f"fuel_{i}" for i in range(10)])

_seed(kb_electrical, [
    "Module voltage should read 13.5-14.8V at idle with alternator charging. Below 13.5V = charging fault.",
    "Battery voltage below 12.0V with engine running is a warning. Below 11.0V causes ECU instability.",
    "A failing alternator shows voltage drop under electrical load (headlights, AC). Check serpentine belt and diodes.",
    "CAN bus errors (U0100) indicate loss of ECU communication — often caused by low battery or wiring faults.",
    "Crankshaft position sensor (CKP) failures cause intermittent no-start and P0335/P0336 codes.",
    "Camshaft position sensor (CMP) failures affect VVT and ignition timing, causing rough running and P0340-P0343.",
    "Ground strap corrosion causes multiple sensor offset errors and electrical gremlins.",
    "Parasitic battery drain above 50mA with accessories off indicates a stuck relay or module failing to sleep.",
    "TPS failures cause hesitation and erratic idle. Signal should rise smoothly 0-5V with throttle.",
    "MAF sensor contamination from oily air filters causes lean conditions. Clean with MAF spray before replacing.",
], [f"elec_{i}" for i in range(10)])

_seed(kb_dtc, [
    "P0300 — Random/Multiple Cylinder Misfire. Causes: worn spark plugs, faulty ignition coils, low fuel pressure, vacuum leaks.",
    "P0171 — System Too Lean (Bank 1). Causes: vacuum leak, dirty MAF sensor, clogged injector, low fuel pressure.",
    "P0420 — Catalyst Efficiency Below Threshold (Bank 1). Often a failing catalytic converter or upstream exhaust leak.",
    "P0128 — Coolant Temp Below Thermostat Regulating Temp. Almost always a stuck-open thermostat.",
    "P0442 — EVAP System Small Leak. Causes: loose fuel cap, cracked EVAP hose, faulty purge valve.",
    "P0401 — EGR Flow Insufficient. Causes: clogged EGR passages, stuck EGR valve, faulty EGR position sensor.",
    "P0340 — Camshaft Position Sensor Circuit Malfunction. Causes stalling and hard starts. Check wiring first.",
    "P0500 — Vehicle Speed Sensor Malfunction. Affects transmission shifts, cruise control, and ABS.",
    "B1000 — Body control module fault. Often linked to battery drain or aftermarket accessories.",
    "U0100 — Lost Communication With ECM/PCM. Check battery voltage, CAN bus wiring, and grounds first.",
], [f"dtc_{i}" for i in range(10)])

print(f"✅ Knowledge base ready: engine={kb_engine.count()}, fuel={kb_fuel.count()}, electrical={kb_electrical.count()}, dtc={kb_dtc.count()}")

✅ Knowledge base ready: engine=10, fuel=10, electrical=10, dtc=10


In [ ]:
# ── CELL 4: SHARED TOOL FUNCTIONS ───────────────────────────
# Four domain-specific search tools used by all agents.

def search_engine_data(query: str, n: int = 3) -> str:
    """
    Search the engine and temperature knowledge base.
    Use for questions about RPM, coolant temp, oil temp, catalyst temp,
    engine load, timing, intake air, and MAF sensor readings.
    """
    results = kb_engine.query(query_texts=[query], n_results=min(n, kb_engine.count()))
    docs = results["documents"][0]
    return "\n---\n".join(docs) if docs else "No engine data found."

def search_fuel_emissions_data(query: str, n: int = 3) -> str:
    """
    Search the fuel system and emissions knowledge base.
    Use for questions about fuel trim (STFT/LTFT), lambda/O2 sensors,
    fuel level, EGR, EVAP, injector balance, and rich/lean conditions.
    """
    results = kb_fuel.query(query_texts=[query], n_results=min(n, kb_fuel.count()))
    docs = results["documents"][0]
    return "\n---\n".join(docs) if docs else "No fuel/emissions data found."

def search_electrical_data(query: str, n: int = 3) -> str:
    """
    Search the electrical systems knowledge base.
    Use for questions about battery voltage, alternator, CAN bus errors,
    throttle position, crankshaft/camshaft sensors, and parasitic drain.
    """
    results = kb_electrical.query(query_texts=[query], n_results=min(n, kb_electrical.count()))
    docs = results["documents"][0]
    return "\n---\n".join(docs) if docs else "No electrical data found."

def search_dtc_data(query: str, n: int = 3) -> str:
    """
    Search the Diagnostic Trouble Code (DTC) knowledge base.
    Use for questions about specific fault codes (P, B, C, U codes),
    their likely causes, and recommended diagnostic steps.
    """
    results = kb_dtc.query(query_texts=[query], n_results=min(n, kb_dtc.count()))
    docs = results["documents"][0]
    return "\n---\n".join(docs) if docs else "No DTC data found."

def _llm(system: str, user: str, max_tokens: int = MAX_TOKENS) -> str:
    """Single LLM call — returns string content."""
    resp = openai_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": user}],
        temperature=TEMPERATURE,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()

print("✅ Shared tools ready.")

✅ Shared tools ready.


In [ ]:
# ── CELL 5: AGENT 1 — ReAct AGENT ──────────────────────────
# Thought → Action → Observation → Thought loop.
# Must complete at least 2 full cycles before giving a Final Answer.

REACT_SYSTEM = """You are an expert OBD-II automotive diagnostic assistant using the ReAct reasoning framework.

You MUST follow this exact pattern for every step — do not skip steps:

Thought: [What you are thinking and why you need information]
Action: [EXACTLY one of: search_engine_data | search_fuel_emissions_data | search_electrical_data | search_dtc_data]
Action Input: [The search query string]
Observation: [The result will be inserted here by the system]

After at least TWO complete Thought-Action-Observation cycles, you may conclude:

Thought: [Your synthesis reasoning]
Final Answer: [Your complete, actionable diagnostic answer]

Rules:
- Always start with a Thought.
- You must complete at least 2 full cycles before giving a Final Answer.
- Action must be one of the four tool names exactly.
- Never make up sensor values — only use what you observe from the tools.
- Be specific: reference actual values, thresholds, and part names.
- Your Final Answer must include: what the problem likely is, urgency level, and next steps.
"""

REACT_TOOLS = {
    "search_engine_data":         search_engine_data,
    "search_fuel_emissions_data":  search_fuel_emissions_data,
    "search_electrical_data":      search_electrical_data,
    "search_dtc_data":             search_dtc_data,
}

def react_agent(query: str, max_iterations: int = 6) -> str:
    """
    ReAct loop: parses Action/Action Input, calls the tool,
    injects Observation, and continues until 'Final Answer:' or cap.
    """
    history = ""
    user_turn = f"Diagnostic question: {query}\n\nBegin your ReAct reasoning chain now."

    for i in range(max_iterations):
        prompt = user_turn if not history else history
        raw = _llm(REACT_SYSTEM, prompt, max_tokens=600)
        history += ("\n" if history else "") + raw

        if "Final Answer:" in raw:
            break

        action_match = re.search(r"Action:\s*(\w+)", raw)
        input_match  = re.search(r"Action Input:\s*(.+)", raw)

        if not action_match or not input_match:
            history += "\nObservation: [Could not parse action — please restate your Action and Action Input clearly.]"
            continue

        action     = action_match.group(1).strip()
        action_inp = input_match.group(1).strip().strip('"').strip("'")
        tool_fn    = REACT_TOOLS.get(action)
        observation = tool_fn(action_inp) if tool_fn else f"Unknown tool '{action}'. Choose from: {list(REACT_TOOLS.keys())}"
        history += f"\nObservation: {observation}"

    return history

# Example queries demonstrating ≥2 cycles each
REACT_EXAMPLE_1 = "My car shows high coolant temperature and the engine feels sluggish. Could this be thermostat-related and is it safe to drive?"
REACT_EXAMPLE_2 = "I have a P0171 code and my long fuel trim is at +18%. What's causing this and what should I check first?"

print(f"✅ Agent 1 (ReAct) ready.")
print(f"Example 1: {REACT_EXAMPLE_1}")
print(f"Example 2: {REACT_EXAMPLE_2}")
print("\nTest it: react_agent(REACT_EXAMPLE_1)")

✅ Agent 1 (ReAct) ready.
Example 1: My car shows high coolant temperature and the engine feels sluggish. Could this be thermostat-related and is it safe to drive?
Example 2: I have a P0171 code and my long fuel trim is at +18%. What's causing this and what should I check first?

Test it: react_agent(REACT_EXAMPLE_1)


In [ ]:
# ── CELL 6: AGENT 2 — DEBATE AGENT ─────────────────────────
# Advocate → Critic → Judge three-agent pipeline.

ADVOCATE_SYSTEM = """You are the Advocate in an OBD-II diagnostic debate panel.
Your role: make the STRONGEST positive case answering the user's question.
Use the knowledge base context provided to build evidence-backed arguments.
Structure your response as:
  POSITION: [Your main claim]
  EVIDENCE: [Facts from the knowledge base supporting your claim]
  REASONING: [Why this evidence supports your position]
Be assertive and specific. Reference sensor thresholds and part names."""

CRITIC_SYSTEM = """You are the Critic in an OBD-II diagnostic debate panel.
Your role: challenge the Advocate's argument, find gaps and alternative explanations.
Use the knowledge base context to find counterevidence or additional complexity.
Structure your response as:
  CHALLENGE: [What the Advocate got wrong or oversimplified]
  COUNTER-EVIDENCE: [Facts that complicate the picture]
  ALTERNATIVE: [A different diagnosis the Advocate ignored]
Be rigorous — a good critic strengthens the final diagnosis."""

JUDGE_SYSTEM = """You are the Judge in an OBD-II diagnostic debate panel.
You receive both the Advocate's argument and the Critic's challenge.
Synthesize both into a balanced, complete final answer.
Structure your response as:
  VERDICT: [Which argument was stronger and why]
  BALANCED FINDING: [The most accurate diagnosis considering both sides]
  ACTION PLAN: [Prioritized steps: what to check first, urgency level, estimated cost range]
Be fair, thorough, and practical."""

def debate_agent(query: str) -> str:
    """Chains Advocate → Critic → Judge with shared knowledge base context."""
    all_context = (
        f"ENGINE DATA:\n{search_engine_data(query)}\n\n"
        f"FUEL DATA:\n{search_fuel_emissions_data(query)}\n\n"
        f"ELECTRICAL DATA:\n{search_electrical_data(query)}\n\n"
        f"DTC DATA:\n{search_dtc_data(query)}"
    )

    # Advocate
    advocate_resp = _llm(ADVOCATE_SYSTEM, f"Question: {query}\n\nContext:\n{all_context}")

    # Critic — receives Advocate's argument + same context
    critic_resp = _llm(CRITIC_SYSTEM,
        f"Question: {query}\n\nAdvocate's argument:\n{advocate_resp}\n\nContext:\n{all_context}")

    # Judge — receives both arguments, no tools
    judge_resp = _llm(JUDGE_SYSTEM,
        f"Original question: {query}\n\nADVOCATE:\n{advocate_resp}\n\nCRITIC:\n{critic_resp}")

    return (f"## 🟢 ADVOCATE\n{advocate_resp}\n\n"
            f"---\n## 🔴 CRITIC\n{critic_resp}\n\n"
            f"---\n## ⚖️ JUDGE\n{judge_resp}")

DEBATE_EXAMPLE = "Should I replace my catalytic converter immediately or could O2 sensors be causing the P0420 code?"

print("✅ Agent 2 (Debate) ready.")
print(f"Example: {DEBATE_EXAMPLE}")
print("\nTest it: debate_agent(DEBATE_EXAMPLE)")

✅ Agent 2 (Debate) ready.
Example: Should I replace my catalytic converter immediately or could O2 sensors be causing the P0420 code?

Test it: debate_agent(DEBATE_EXAMPLE)


In [ ]:
# ── CELL 7: AGENT 3 — SUPERVISOR AGENT ─────────────────────
# Draft → evaluate_answer (APPROVED/RETRY) → retry loop.

SUPERVISOR_SYSTEM = """You are a senior OBD-II diagnostic specialist with a self-evaluation discipline.

Draft a diagnostic answer using the knowledge base context provided.
Your answer must include:
  - The most likely root cause with evidence
  - Supporting sensor values or DTC descriptions
  - Urgency level: STOP DRIVING / SCHEDULE SOON / MONITOR
  - Recommended next diagnostic steps"""

EVALUATOR_SYSTEM = """You are a quality evaluator for OBD-II diagnostic answers.
Assess whether the answer:
  1. Directly addresses the question with a specific diagnosis (not vague)
  2. References actual technical evidence (thresholds, codes, failure patterns)
  3. States a clear urgency level
  4. Gives actionable next steps

Respond with EXACTLY one of:
  APPROVED: [brief reason why the answer is complete]
  RETRY: [specific feedback on what is missing]

Output nothing else."""

def evaluate_answer(question: str, draft: str) -> str:
    """
    LLM evaluator — returns 'APPROVED: ...' or 'RETRY: ...'
    with specific feedback about what is missing or needs improvement.
    """
    return _llm(EVALUATOR_SYSTEM, f"QUESTION: {question}\n\nDRAFT ANSWER:\n{draft}", max_tokens=200)

def supervisor_agent(query: str, max_retries: int = 3) -> str:
    """
    Self-evaluating loop: searches, drafts, evaluates, and retries
    with different search angles if evaluator returns RETRY.
    """
    search_angles = [
        lambda q: f"ENGINE: {search_engine_data(q)}\nFUEL: {search_fuel_emissions_data(q)}",
        lambda q: f"ELECTRICAL: {search_electrical_data(q)}\nDTC: {search_dtc_data(q)}",
        lambda q: (f"ENGINE: {search_engine_data(q)}\nFUEL: {search_fuel_emissions_data(q)}\n"
                   f"ELECTRICAL: {search_electrical_data(q)}\nDTC: {search_dtc_data(q)}"),
    ]

    log = []
    draft = ""
    for attempt in range(max_retries):
        context    = search_angles[attempt % len(search_angles)](query)
        retry_hint = log[-1] if log else ""
        prompt = (
            f"Diagnostic question: {query}\n\nContext:\n{context}\n\n"
            + (f"Previous evaluation feedback to address: {retry_hint}\n\n" if retry_hint else "")
            + "Provide your diagnostic answer."
        )
        draft      = _llm(SUPERVISOR_SYSTEM, prompt)
        evaluation = evaluate_answer(query, draft)
        log.append(f"Attempt {attempt+1}: {evaluation}")

        if evaluation.startswith("APPROVED"):
            return (f"**Attempts:** {attempt+1}\n**Evaluation:** {evaluation}\n\n"
                    f"---\n## ✅ Final Answer\n{draft}\n\n"
                    f"---\n*Evaluation log:*\n" + "\n".join(log))

    return (f"**Attempts:** {max_retries} (max reached)\n\n"
            f"---\n## ⚠️ Best Answer (after {max_retries} attempts)\n{draft}\n\n"
            f"---\n*Log:*\n" + "\n".join(log))

SUPERVISOR_EXAMPLE = "My battery keeps dying overnight but the alternator tested fine. What's draining it?"

print("✅ Agent 3 (Supervisor) ready.")
print(f"Example (triggers retry): {SUPERVISOR_EXAMPLE}")
print("\nTest it: supervisor_agent(SUPERVISOR_EXAMPLE)")

✅ Agent 3 (Supervisor) ready.
Example (triggers retry): My battery keeps dying overnight but the alternator tested fine. What's draining it?

Test it: supervisor_agent(SUPERVISOR_EXAMPLE)


In [ ]:
# ── CELL 8: AGENT 4 — MULTI-AGENT PARALLEL SYSTEM ──────────
# Three specialist agents run simultaneously via asyncio.gather().
# A synthesizer combines all three reports.

import nest_asyncio
nest_asyncio.apply()   # Required for asyncio in Colab

SPECIALIST_PROMPTS = {
    "Engine & Temperature Specialist": (
        "You are an OBD-II engine and temperature diagnostic specialist. "
        "Analyze the question using ONLY the ENGINE context provided. "
        "Report: what engine/temperature factors are relevant, any warning signs, and your specialist finding."
    ),
    "Fuel & Emissions Specialist": (
        "You are an OBD-II fuel system and emissions diagnostic specialist. "
        "Analyze the question using ONLY the FUEL/EMISSIONS context provided. "
        "Report: what fuel or emissions factors are relevant, any anomalies, and your specialist finding."
    ),
    "Electrical & DTC Specialist": (
        "You are an OBD-II electrical systems and fault code diagnostic specialist. "
        "Analyze the question using ONLY the ELECTRICAL/DTC context provided. "
        "Report: what electrical issues or DTCs are relevant, wiring concerns, and your specialist finding."
    ),
}

SYNTHESIZER_SYSTEM = """You are the lead OBD-II diagnostic synthesizer.
You receive reports from three specialist agents. Your job:
1. Identify where specialists agree — these are high-confidence findings.
2. Identify where specialists differ — explain which is more likely.
3. Produce a unified diagnosis with a prioritized action plan.
4. Assign an overall urgency: STOP DRIVING / SCHEDULE SOON / MONITOR.
Do not search for additional information — synthesize only what the specialists found."""

async def _run_specialist(name: str, system: str, query: str, context: str) -> tuple:
    """Async wrapper: runs a single specialist LLM call in a thread executor."""
    loop = asyncio.get_event_loop()
    user_msg = f"Diagnostic question: {query}\n\nYour domain context:\n{context}"
    report = await loop.run_in_executor(None, lambda: _llm(system, user_msg))
    return name, report

async def _parallel_specialists(query: str) -> dict:
    """Launch all three specialists simultaneously and collect results."""
    contexts = {
        "Engine & Temperature Specialist":  search_engine_data(query),
        "Fuel & Emissions Specialist":      search_fuel_emissions_data(query),
        "Electrical & DTC Specialist":      f"{search_electrical_data(query)}\n{search_dtc_data(query)}",
    }
    tasks = [
        _run_specialist(name, SPECIALIST_PROMPTS[name], query, contexts[name])
        for name in SPECIALIST_PROMPTS
    ]
    # asyncio.gather runs all three simultaneously
    results = await asyncio.gather(*tasks)
    return dict(results)

def parallel_agent(query: str) -> str:
    """
    Entry point: runs specialists IN PARALLEL via asyncio.gather(),
    then calls synthesizer to combine findings.
    """
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    specialist_reports = loop.run_until_complete(_parallel_specialists(query))

    combined = "\n\n".join(
        f"=== {name} ===\n{report}" for name, report in specialist_reports.items()
    )
    synthesis = _llm(SYNTHESIZER_SYSTEM, f"Question: {query}\n\nSpecialist reports:\n{combined}")

    icons = {"Engine & Temperature Specialist": "🔥",
             "Fuel & Emissions Specialist":     "⛽",
             "Electrical & DTC Specialist":     "⚡"}
    parts = ["## 🔬 SPECIALIST REPORTS (ran in parallel)\n"]
    for name, report in specialist_reports.items():
        parts.append(f"### {icons.get(name,'')} {name}\n{report}\n")
    parts.append(f"---\n## 🧬 SYNTHESIZER\n{synthesis}")
    return "\n".join(parts)

PARALLEL_EXAMPLE = "My car misfires at highway speed, has a P0300 code, and the battery warning light flickered. What's going on?"

print("✅ Agent 4 (Parallel) ready.")
print(f"Example (needs 2+ specialists): {PARALLEL_EXAMPLE}")
print("\nTest it: parallel_agent(PARALLEL_EXAMPLE)")

✅ Agent 4 (Parallel) ready.
Example (needs 2+ specialists): My car misfires at highway speed, has a P0300 code, and the battery warning light flickered. What's going on?

Test it: parallel_agent(PARALLEL_EXAMPLE)


In [ ]:
# ── CELL 9: AGENT 5 — META-INTELLIGENT ROUTER ──────────────

# ── GUARDRAILS ───────────────────────────────────────────────

# Input guardrails — check BEFORE routing to any agent

INPUT_BLOCKED_PATTERNS = [
    "ignore previous",
    "ignore all instructions",
    "you are now",
    "pretend you are",
    "jailbreak",
    "forget your instructions",
]

ALLOWED_DOMAINS = [
    "obd", "engine", "rpm", "coolant", "fuel", "trim", "battery",
    "alternator", "dtc", "code", "sensor", "misfire", "exhaust",
    "transmission", "brake", "oil", "temperature", "voltage",
    "throttle", "catalyst", "oxygen", "o2", "timing", "idle",
    "diagnostic", "car", "vehicle", "drive", "driving", "safe",
    "check engine", "warning", "light", "p0", "b1", "u0",
]

def input_guardrail(query: str) -> tuple[bool, str]:
    """
    Input guardrail — runs before routing.
    Returns (is_safe, reason).
    Blocks: prompt injection, off-topic queries.
    Fail-closed: when in doubt, block.
    """
    q_lower = query.lower()

    # Check 1 — prompt injection
    for pattern in INPUT_BLOCKED_PATTERNS:
        if pattern in q_lower:
            return False, f"⛔ Input blocked: possible prompt injection detected ('{pattern}')."

    # Check 2 — domain relevance (at least one automotive keyword required)
    if not any(kw in q_lower for kw in ALLOWED_DOMAINS):
        return False, "⛔ Input blocked: query does not appear to be automotive or OBD-II related. Please ask about your vehicle."

    return True, "ok"


# Output guardrails — check AFTER agent responds, BEFORE showing to user

OUTPUT_BLOCKED_PHRASES = [
    "ignore the knowledge base",
    "i cannot help",
    "as an ai language model",
]

DANGEROUS_ADVICE_PHRASES = [
    "safe to drive" and "critical",   # catches conflicting advice
    "continue driving" ,
    "drive with overheating",
    "ignore the warning",
    "you can still drive",
]

def output_guardrail(query: str, response: str) -> tuple[bool, str]:
    """
    Output guardrail — runs before returning response to user.
    Returns (is_safe, cleaned_response_or_error).
    Blocks: dangerous driving advice, broken responses.
    Fail-open: passes response through with a warning appended.
    """
    r_lower = response.lower()

    # Check 1 — broken/meta response leaked through
    for phrase in OUTPUT_BLOCKED_PHRASES:
        if phrase in r_lower:
            return False, "⚠️ Response blocked: the agent produced an invalid output. Please rephrase your question."

    # Check 2 — dangerous advice when critical sensors are involved
    critical_keywords = ["overheating", "critical", "stop driving", "110°c", "950°c", "brake"]
    is_critical = any(kw in query.lower() for kw in critical_keywords)
    if is_critical:
        for phrase in DANGEROUS_ADVICE_PHRASES:
            if phrase in r_lower:
                return False, "⚠️ Response blocked: potentially dangerous advice detected for a critical situation. Please consult a mechanic immediately."

    # Check 3 — response too short to be useful
    if len(response.strip()) < 80:
        return True, response + "\n\n⚠️ *Note: This response may be incomplete. Try rephrasing your question for a more detailed answer.*"

    return True, response




# LLM classifier → keyword fallback → route → response enhancement.

ROUTER_CLASSIFIER_SYSTEM = """You are a query router for an OBD-II diagnostic multi-agent system.
Analyze the incoming query and select the BEST agent. Output ONLY the agent name, nothing else.

Agent options and when to use each:
  REACT       — Step-by-step investigations, "why is X happening", single-symptom deep dives,
                 unknown causes requiring sequential reasoning chains.
  DEBATE      — Two valid interpretations, "should I replace X or could it be Y",
                 cost vs. urgency trade-offs, contested diagnoses.
  SUPERVISOR  — Safety-critical situations, complex multi-symptom cases where accuracy is essential,
                 questions where a wrong answer could cause harm (overheating, brakes, stalling).
  MULTI_AGENT — Queries spanning multiple vehicle systems, multiple DTCs simultaneously,
                 broad "what's wrong with my car" questions.

Output EXACTLY one of: REACT, DEBATE, SUPERVISOR, MULTI_AGENT"""

ENHANCER_SYSTEM = """You are a response editor for an automotive diagnostic assistant.
Polish the diagnostic response below for clarity without adding new technical claims.
Ensure:
  - Starts with a clear one-sentence summary
  - Technical terms are briefly explained on first use
  - Urgency is stated explicitly at the end
  - Tone is professional but accessible to a non-mechanic car owner
Keep the same information — only improve structure and clarity."""

KEYWORD_FALLBACK = {
    "REACT":       ["why", "reason", "cause", "explain", "how does", "what is", "diagnose", "investigate"],
    "DEBATE":      ["should i", "replace or", "worth it", "better to", "is it", "could it be", "or could"],
    "SUPERVISOR":  ["safe to drive", "critical", "overheating", "brake", "stall", "dangerous", "urgent", "keep dying"],
    "MULTI_AGENT": ["multiple", "several", "many codes", "and also", "plus", "combination", "everything", "whole car"],
}

# Routing table: one example query per agent with explanation
ROUTING_TABLE = {
    "REACT":       ("Why is my idle RPM so high when the engine is fully warmed up?",
                    "Needs sequential investigation: check IAC, vacuum leaks, then fuel trim"),
    "DEBATE":      ("Should I replace my catalytic converter immediately or could O2 sensors be causing P0420?",
                    "Two valid diagnoses with different costs — needs Advocate/Critic/Judge"),
    "SUPERVISOR":  ("My brakes feel spongy and the ABS light is on — is it safe to drive?",
                    "Safety-critical — needs verified, evaluated answer before driver acts"),
    "MULTI_AGENT": ("I have a P0300 misfire, P0171 lean code, and my battery light came on all at once.",
                    "Spans engine + fuel + electrical simultaneously — needs parallel specialists"),
}

def _classify_query(query: str) -> str:
    """LLM-based classifier. Returns REACT/DEBATE/SUPERVISOR/MULTI_AGENT or None on failure."""
    try:
        result = _llm(ROUTER_CLASSIFIER_SYSTEM, f"Query: {query}", max_tokens=10).strip().upper()
        if result in ("REACT", "DEBATE", "SUPERVISOR", "MULTI_AGENT"):
            return result
    except Exception:
        pass
    return None

def _keyword_fallback(query: str) -> str:
    """Keyword fallback when LLM classifier fails or returns invalid output."""
    q = query.lower()
    scores = {agent: sum(1 for kw in kws if kw in q) for agent, kws in KEYWORD_FALLBACK.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "REACT"

def _enhance_response(query: str, response: str) -> str:
    """Polish the routed agent's response for clarity and completeness."""
    return _llm(ENHANCER_SYSTEM, f"Original question: {query}\n\nResponse to polish:\n{response}")

def meta_agent(query: str) -> str:
    # ── INPUT GUARDRAIL ──────────────────────────────────────
    is_safe, reason = input_guardrail(query)
    if not is_safe:
        return reason

    # ── ROUTE ────────────────────────────────────────────────
    agent_name = _classify_query(query)
    method = "LLM classifier"
    if agent_name is None:
        agent_name = _keyword_fallback(query)
        method = "keyword fallback"

    if agent_name == "REACT":
        raw = react_agent(query)
    elif agent_name == "DEBATE":
        raw = debate_agent(query)
    elif agent_name == "SUPERVISOR":
        raw = supervisor_agent(query)
    else:
        raw = parallel_agent(query)

    enhanced = _enhance_response(query, raw)

    # ── OUTPUT GUARDRAIL ─────────────────────────────────────
    is_safe, final_response = output_guardrail(query, enhanced)
    if not is_safe:
        return final_response

    routing_header = f"🧭 **Routed to:** `{agent_name}` via {method}\n\n---\n"
    return routing_header + final_response

print("✅ Agent 5 (Meta Router) ready.")
print("\nRouting table:")
for agent, (q, reason) in ROUTING_TABLE.items():
    print(f"  {agent}: '{q}'")
    print(f"    → {reason}")
print("\nTest it: meta_agent('My car misfires and the battery light is on')")

✅ Agent 5 (Meta Router) ready.

Routing table:
  REACT: 'Why is my idle RPM so high when the engine is fully warmed up?'
    → Needs sequential investigation: check IAC, vacuum leaks, then fuel trim
  DEBATE: 'Should I replace my catalytic converter immediately or could O2 sensors be causing P0420?'
    → Two valid diagnoses with different costs — needs Advocate/Critic/Judge
  SUPERVISOR: 'My brakes feel spongy and the ABS light is on — is it safe to drive?'
    → Safety-critical — needs verified, evaluated answer before driver acts
  MULTI_AGENT: 'I have a P0300 misfire, P0171 lean code, and my battery light came on all at once.'
    → Spans engine + fuel + electrical simultaneously — needs parallel specialists

Test it: meta_agent('My car misfires and the battery light is on')


In [ ]:
# ── CELL 10: LAUNCH ALL AGENTS ──────────────────────────────
# Tabbed Gradio interface. Each tab can be tested independently.
# The Meta Router tab routes automatically to the best agent.

with gr.Blocks(title="OBD-II Agentic RAG — Final Exam") as full_demo:
    gr.Markdown(
        "# 🔧 OBD-II Agentic RAG System — Final Exam\n"
        "*Five agent architectures on a shared automotive knowledge base.*\n\n"
        "| Agent | Architecture | Best for |\n"
        "|-------|-------------|----------|\n"
        "| 1 ReAct | Thought→Action→Observation | Step-by-step root cause investigation |\n"
        "| 2 Debate | Advocate→Critic→Judge | Contested or uncertain diagnoses |\n"
        "| 3 Supervisor | Draft→Evaluate→Retry | Safety-critical, verified answers |\n"
        "| 4 Parallel | 3 specialists + synthesizer | Multi-system, multi-code diagnoses |\n"
        "| 5 Meta Router | LLM classifier + all 4 agents | Any query — routes automatically |"
    )

    with gr.Tabs():

        # ── Agent 1: ReAct ──────────────────────────────────────
        with gr.Tab("🔍 Agent 1: ReAct"):
            gr.Markdown(
                "### ReAct Agent — Thought → Action → Observation\n"
                "Shows explicit reasoning chain before every tool call.\n\n"
                f"**Try:** *{REACT_EXAMPLE_1}*\n\n"
                f"**Try:** *{REACT_EXAMPLE_2}*"
            )
            react_cb  = gr.Chatbot(height=420, type="messages")
            react_tb  = gr.Textbox(placeholder="Ask a diagnostic question...", show_label=False)
            react_btn = gr.Button("Send", variant="primary")

            def _react(msg, hist):
                hist = hist or []
                hist.append({"role": "user",      "content": msg})
                hist.append({"role": "assistant",  "content": react_agent(msg)})
                return hist, ""

            react_btn.click(_react, [react_tb, react_cb], [react_cb, react_tb])
            react_tb.submit(_react, [react_tb, react_cb], [react_cb, react_tb])

        # ── Agent 2: Debate ─────────────────────────────────────
        with gr.Tab("⚖️ Agent 2: Debate"):
            gr.Markdown(
                "### Debate Agent — Advocate → Critic → Judge\n"
                "Three agents debate the diagnosis; a Judge delivers the verdict.\n\n"
                f"**Try:** *{DEBATE_EXAMPLE}*"
            )
            debate_cb  = gr.Chatbot(height=420, type="messages")
            debate_tb  = gr.Textbox(placeholder="Ask a diagnostic question...", show_label=False)
            debate_btn = gr.Button("Send", variant="primary")

            def _debate(msg, hist):
                hist = hist or []
                hist.append({"role": "user",      "content": msg})
                hist.append({"role": "assistant",  "content": debate_agent(msg)})
                return hist, ""

            debate_btn.click(_debate, [debate_tb, debate_cb], [debate_cb, debate_tb])
            debate_tb.submit(_debate, [debate_tb, debate_cb], [debate_cb, debate_tb])

        # ── Agent 3: Supervisor ─────────────────────────────────
        with gr.Tab("🔄 Agent 3: Supervisor"):
            gr.Markdown(
                "### Supervisor Agent — Draft → Evaluate → Retry\n"
                "Evaluates its own answers and retries with different searches if rejected.\n\n"
                f"**Try (triggers retry):** *{SUPERVISOR_EXAMPLE}*"
            )
            sup_cb  = gr.Chatbot(height=420, type="messages")
            sup_tb  = gr.Textbox(placeholder="Ask a diagnostic question...", show_label=False)
            sup_btn = gr.Button("Send", variant="primary")

            def _supervisor(msg, hist):
                hist = hist or []
                hist.append({"role": "user",      "content": msg})
                hist.append({"role": "assistant",  "content": supervisor_agent(msg)})
                return hist, ""

            sup_btn.click(_supervisor, [sup_tb, sup_cb], [sup_cb, sup_tb])
            sup_tb.submit(_supervisor, [sup_tb, sup_cb], [sup_cb, sup_tb])

        # ── Agent 4: Parallel ───────────────────────────────────
        with gr.Tab("⚡ Agent 4: Parallel"):
            gr.Markdown(
                "### Multi-Agent Parallel System\n"
                "Engine, Fuel, and Electrical specialists run **simultaneously** via `asyncio.gather()`. "
                "A synthesizer combines their findings.\n\n"
                f"**Try:** *{PARALLEL_EXAMPLE}*"
            )
            par_cb  = gr.Chatbot(height=420, type="messages")
            par_tb  = gr.Textbox(placeholder="Ask a diagnostic question...", show_label=False)
            par_btn = gr.Button("Send", variant="primary")

            def _parallel(msg, hist):
                hist = hist or []
                hist.append({"role": "user",      "content": msg})
                hist.append({"role": "assistant",  "content": parallel_agent(msg)})
                return hist, ""

            par_btn.click(_parallel, [par_tb, par_cb], [par_cb, par_tb])
            par_tb.submit(_parallel, [par_tb, par_cb], [par_cb, par_tb])

        # ── Agent 5: Meta Router ────────────────────────────────
        with gr.Tab("🧠 Agent 5: Meta Router ⭐"):
            gr.Markdown(
                "### Meta-Intelligent Router\n"
                "LLM classifies your query → routes to best agent → polishes the response.\n\n"
                "**Routing table (one example per route):**\n"
                + "\n".join(f"- **{a}:** *{q}*" for a, (q, _) in ROUTING_TABLE.items())
            )
            meta_cb  = gr.Chatbot(height=420, type="messages")
            meta_tb  = gr.Textbox(placeholder="Ask anything about your vehicle...", show_label=False)
            meta_btn = gr.Button("Send", variant="primary")

            def _meta(msg, hist):
                hist = hist or []
                hist.append({"role": "user",      "content": msg})
                hist.append({"role": "assistant",  "content": meta_agent(msg)})
                return hist, ""

            meta_btn.click(_meta, [meta_tb, meta_cb], [meta_cb, meta_tb])
            meta_tb.submit(_meta, [meta_tb, meta_cb], [meta_cb, meta_tb])

full_demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e9bf7956ccfa5e28be.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
